# Additional evaluation: MAUVE and a third LLM judge (Colab, GPU)

Computes MAUVE between the generations and the held-out reference continuations of every run, and re-rates the stratified LLM-judge sample with Phi-4 (14B, 4-bit), a judge of a family different from both generators. No new generation is needed: the inputs are the results already in Google Drive. Runtime on a T4: about 30 min for MAUVE and about 1 h for the judge.

In [ ]:
!nvidia-smi -L
!git clone -b paper https://github.com/alexroehrs/dynalora.git /content/dynalora
%cd /content/dynalora
!git pull -q
!pip install -q -r requirements.txt mauve-text faiss-cpu bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; print(os.listdir('/content/drive/MyDrive/dynalora_results'))

## 1. MAUVE against the reference texts

In [ ]:
!python scripts/mauve_eval.py --inputs "/content/drive/MyDrive/dynalora_results/*/generations_all_metrics.csv" --out /content/drive/MyDrive/dynalora_results/mauve

## 2. Third LLM judge (Phi-4) on the same stratified sample

In [ ]:
!python scripts/llm_judge.py --inputs /content/drive/MyDrive/dynalora_results/gpt2/generations_all_metrics.csv /content/drive/MyDrive/dynalora_results/qwen/generations_all_metrics.csv --out /content/drive/MyDrive/dynalora_results/judge --judges microsoft/phi-4 --batch_size 8

## 3. Package the new results (scores only, no reference text)

In [ ]:
!cd /content/drive/MyDrive/dynalora_results && zip -qr /content/dynalora_extra_eval.zip mauve judge/scores__phi-4.csv judge/judge_summary.csv
from google.colab import files
files.download('/content/dynalora_extra_eval.zip')